# School-level variables, 2020-21 (school locations used by 03)

- **Paper:** (prep)
- **Original notebook:** `5_test_scores_explore.ipynb`
- **Reads:**
  - `outputs/enrollment_by_school.pkl`
  - `data/derived/school_AP_IB_courses.pkl (see data/README.md)`
  - `data/raw/school_related/...`
- **Writes:**
  - `outputs/school_related.pkl`


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import requests
import zipfile
# pip install contextily  # see requirements.txt
import contextily as ctx





In [ ]:
# reading data



# enrollment
# with open(f'{OUTPUT_DIR}/enrollment_by_school.pkl','rb') as f:
#   college_enrollment = pickle.load(f)

college_enrollment = pd.read_pickle(f'{OUTPUT_DIR}/enrollment_by_school.pkl')
#f'{OUTPUT_DIR}/svi_enrolled_all_types.pkl'
# math/ english test
#sbpth = f'{DATA_DIR}/school_related/test_scores/sb_ca2019_all_csv_v4.txt'


#sb_test = pd.read_csv(sbpth,sep = ',',encoding='latin-1')
sb_test = pd.read_csv(f'{DATA_DIR}/school_related/20-21/sb_ca2021_1_csv_v2.txt',sep ="^",encoding = 'latin-1')

sb_entities = pd.read_csv(f'{DATA_DIR}/school_related/test_scores/sb_ca2021entities_csv.txt',sep = '^',encoding='latin-1')
# science test
cast_entities = pd.read_csv(f'{DATA_DIR}/school_related/test_scores/cast_ca2021entities_csv.txt',sep = "^",encoding='latin-1')
cast_test = pd.read_csv(f'{DATA_DIR}/school_related/test_scores/cast_ca2021_1_csv_v2.txt',sep = '^',encoding='latin-1')

#cast_test = pd.read_csv(pth,sep =",",encoding = 'latin-1')

# absences
absent_file = f"{DATA_DIR}/school_related/absent_2021.txt"
absent = pd.read_csv(absent_file,sep = '\t',encoding='latin-1')

# teacher experience
"""
can only find staff demographic data for 2018-2019 and before
but it is much more detailed than the assignment monitoring outcome data
"""

staff_demo = pd.read_csv(f'{DATA_DIR}/school_related/StaffDemo18.txt',sep = '\t',encoding='latin-1')
staff_cred = pd.read_csv(f'{DATA_DIR}/school_related/StaffCred18.txt',sep = '\t',encoding='latin-1')
staff_FTE = pd.read_csv(f'{DATA_DIR}/school_related/StaffSchoolFTE18.txt',sep = '\t',encoding='latin-1')

# free and reduced lunch
lunch_file = f"{DATA_DIR}/school_related/free_reduced_2021.xlsx"
lunch = pd.read_excel(lunch_file, sheet_name = "FRPM School-Level Data ")

# per pupil spending

"""
https://www.cde.ca.gov/fg/ac/es/essappedata.asp
DNR = Did Not Report; LEA = Local Education Agency; CDS = County District School
"""
spending = pd.read_excel(f'{DATA_DIR}/school_related/essappe2021data.xlsx', skiprows=6,sheet_name="School Level Data")

# census tracts
tracts = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/ca_tracts_2022')

# course offerings
courses =pd.read_pickle(f"{DERIVED_DIR}/school_AP_IB_courses.pkl")


In [ ]:
courses = courses.rename(columns = {"schoolCode":"SchoolCode"})
courses = courses[['SchoolCode',"IB_Count","AP_Count"]]

In [ ]:
# subsetting to schoolwide data/ taking columns we want
#college_enrollment.columns
college_enrollment = college_enrollment[['SchoolCode','SchoolName','CountyName','College Going Rate - Total (12 Months)','Latitude',"Longitude",'High School Completers','Enrolled In College - Total (12 Months)']]
college_enrollment = college_enrollment.rename(columns = {'College Going Rate - Total (12 Months)':'college_going_rate','High School Completers':'hs_completers','Enrolled In College - Total (12 Months)':'enrolled_college'})

absent = absent[absent['Aggregate Level']=='S']
absent = absent[absent['Reporting Category']=='TA']
absent.columns
absent = absent[['School Code','ChronicAbsenteeismRate', 'ChronicAbsenteeismEligibleCumula', 'ChronicAbsenteeismCount']]

spending['school_expenditures'] = spending['School Expenditures - Federal']+spending['School Expenditures - State & Local']
spending = spending[['School Name','County','school_expenditures']]
lunch.columns
lunch = lunch[['School Code','Percent (%) \nEligible FRPM \n(K-12)','FRPM Count \n(K-12)', 'Enrollment \n(K-12)']]
lunch = lunch.rename(columns = {'Percent (%) \nEligible FRPM \n(K-12)':'pct_frpm','FRPM Count \n(K-12)':'frpm_count', 'Enrollment \n(K-12)':'n_students'})

In [ ]:
sb_test.columns.values

In [ ]:
# smarter balanced assessment cleaning
#sb_test = sb_test[sb_test['Type ID']==7]
sb_test['Percentage Standard Met and Above'] = pd.to_numeric(sb_test['Percentage Standard Met and Above'], errors='coerce')

sb_test = sb_test[sb_test['Student Group ID']==1]

sb_test = sb_test[sb_test['Grade']==13]
sb_test['School Code'] = pd.to_numeric(sb_test['School Code'], errors='coerce')


sb_test = sb_test[(sb_test['School Code'] > 0) & (~sb_test['School Code'].isna())]


sb_test_math = sb_test[sb_test['Test ID']==2]
sb_test_math = sb_test_math[['School Code','Percentage Standard Met and Above',]]

sb_test_math = sb_test_math.rename(columns = {'Percentage Standard Met and Above':'pct_passed_math'})

sb_test_english = sb_test[sb_test['Test ID']==1]
sb_test_english = sb_test_english[['School Code','Percentage Standard Met and Above']]

sb_test_english = sb_test_english.rename(columns = {'Percentage Standard Met and Above':'pct_passed_english'})

sb_test = pd.merge(sb_test_math,sb_test_english,on = 'School Code')
sb_test= sb_test.dropna()

# sb_test = sb_test.groupby('School Code')['sb_mean_score'].mean().reset_index()




In [ ]:
# CAST cleaning

#cast_test = cast_test[cast_test['Type ID']==7]
cast_test['Percentage Standard Met and Above'] = pd.to_numeric(cast_test['Percentage Standard Met and Above'], errors='coerce')

cast_test = cast_test[cast_test['Student Group ID']==1]
cast_test.columns
cast_test = cast_test[(cast_test['Grade']==14) ]

cast_test['School Code'] = pd.to_numeric(cast_test['School Code'], errors='coerce')

# cast_test = cast_test[(cast_test['School Code'] > 0) & (~cast_test['School Code'].isna())]
cast_test = cast_test[['School Code','Percentage Standard Met and Above']]
cast_test.rename(columns={'Percentage Standard Met and Above': 'pct_passed_science'}, inplace=True)

cast_test= cast_test.dropna()


In [ ]:
# teacher experience cleaning
staff_demo = staff_demo.drop(columns = ["AcademicYear","FileCreated","FTE Teaching","FTE administrative","FTE PupilServices"])

#staff_demo = pd.merge(staff_demo,staff_cred,on = ['RecID','AcademicYear'],how = 'left')
staff_demo = pd.merge(staff_demo,staff_FTE,on = ['RecID'],how = 'left')

"""
D = Doctorate
S = Special
V = Master's degree plus 30 or more semester hours
M = Master's degree
U = Fifth year within bachelor's degree
Y = Fifth year induction
F = Fifth year
C = Baccalaureate plus 30 or more semester hours
B = Baccalaureate
A = Associate degree
N = Not reported
"""
staff_demo['BeyondBachelors'] = staff_demo['EducationLevel'].isin(['D', 'S', 'V', 'M'])
bins =[0,3,10,float('inf')]

staff_demo['experience_level'] = pd.cut(staff_demo['YearsTeaching'],bins = bins,labels = [1,2,3],right=False)
staff_demo = pd.get_dummies(staff_demo, columns=['experience_level'], prefix='TeachingExp')


staff_demo= staff_demo[staff_demo['StaffType']=='T']
total_teachers = staff_demo.groupby('SchoolCode').size()
staff_demo["years_teaching_std"] = staff_demo['YearsTeaching']
teacher_experience =staff_demo.groupby('SchoolCode').agg({

    'YearsTeaching': 'mean',
    'years_teaching_std':'std'
   # ,'TeachingExp_1':'mean',
   # 'TeachingExp_2':'mean',
   # 'TeachingExp_3':'mean'
}).reset_index()

teacher_experience['n_teachers'] = total_teachers.values
# other data already in schoolwide form
teacher_experience

In [ ]:
merged_df = pd.merge(college_enrollment,teacher_experience,on = ['SchoolCode'],how ='left')
merged_df = pd.merge(merged_df,courses,on = 'SchoolCode')

merged_df = pd.merge(merged_df,absent,left_on="SchoolCode",right_on ='School Code',how ='left')
merged_df = merged_df.drop(columns = 'School Code')

merged_df = pd.merge(merged_df,lunch,left_on="SchoolCode",right_on ='School Code',how ='left')
merged_df = merged_df.drop(columns = 'School Code')

merged_df = pd.merge(merged_df,cast_test,left_on="SchoolCode",right_on ='School Code',how ='left')
merged_df = merged_df.drop(columns = 'School Code')
merged_df = pd.merge(merged_df,sb_test,left_on="SchoolCode",right_on ='School Code',how ='left')
merged_df = merged_df.drop(columns = 'School Code')

merged_df = pd.merge(merged_df, spending, left_on= ["SchoolName",'CountyName'],right_on = ["School Name","County"],how = 'left')
merged_df = merged_df.drop(columns = ['School Name','County'])

with open(f'{OUTPUT_DIR}/school_related.pkl','wb') as f:
  pickle.dump(merged_df,f)


In [ ]:
merged_df.columns

In [ ]:
relevant  = merged_df.drop(columns = ['SchoolCode','SchoolName','CountyName',
                                      'n_teachers',"Latitude",'Longitude',
                                      'n_students','frpm_count',
                                      "ChronicAbsenteeismCount",'hs_completers',
                                       'ChronicAbsenteeismEligibleCumula'])
for col in relevant.columns:
    relevant[col] = pd.to_numeric(relevant[col], errors='coerce')


In [ ]:
sns.pairplot(relevant)
plt.show()

In [ ]:
cor_mat = relevant.corr(numeric_only = True)
matrix = np.triu(cor_mat)

plt.figure(figsize=(12, 8))
sns.heatmap(cor_mat, annot=True, cmap='coolwarm', fmt=".2f",mask=matrix)
plt.title('Correlation Matrix')
plt.show()

In [ ]:

# Set marker properties
marker_size = 50
marker_color = 'black'
marker_edge_color = 'white'
marker_edge_width = 0.5

# FRPM Percent vs. College Going Rate
plt.figure(figsize=(8, 6))
plt.scatter(relevant['college_going_rate'],
            relevant['pct_frpm'],
            s=marker_size, c=marker_color, edgecolors=marker_edge_color, linewidths=marker_edge_width)
plt.title('College Going Rate vs. FRPM Percent')
plt.xlabel('College Going Rate')
plt.ylabel('FRPM Percent')
plt.savefig(f'{OUTPUT_DIR}/plots/frpm_vs_college.png', dpi=500)
plt.close()

# SB Mean Score vs. College Going Rate
plt.figure(figsize=(8, 6))
plt.scatter(relevant['college_going_rate'],
            relevant['pct_passed_math'],
            s=marker_size, c=marker_color, edgecolors=marker_edge_color, linewidths=marker_edge_width)
plt.title('College Going Rate vs. Smarter Balanced Assessment Mean Score')
plt.xlabel('College Going Rate')
plt.ylabel('SB Mean Score')
plt.savefig(f'{OUTPUT_DIR}/plots/sb_mean_score_vs_college.png', dpi=500)
plt.close()

# CAST Mean Score vs. College Going Rate
plt.figure(figsize=(8, 6))
plt.scatter(relevant['college_going_rate'],
            relevant['pct_passed_science'],
            s=marker_size, c=marker_color, edgecolors=marker_edge_color, linewidths=marker_edge_width)
plt.title('College Going Rate vs. CAST Mean Score')
plt.xlabel('College Going Rate')
plt.ylabel('CAST Mean Score')
plt.savefig(f'{OUTPUT_DIR}/plots/cast_mean_score_vs_college.png', dpi=500)
plt.close()

# Chronic Absenteeism Rate vs. College Going Rate
plt.figure(figsize=(8, 6))
plt.scatter(relevant['college_going_rate'],
            relevant['ChronicAbsenteeismRate'],
            s=marker_size, c=marker_color, edgecolors=marker_edge_color, linewidths=marker_edge_width)
plt.title('College Going Rate vs. Chronic Absenteeism Rate')
plt.xlabel('College Going Rate')
plt.ylabel('Chronic Absenteeism Rate')
plt.savefig(f'{OUTPUT_DIR}/plots/chronic_absenteeism_rate_vs_college.png', dpi=500)
plt.close()


In [ ]:
merged_df = merged_df[['Latitude',"Longitude",'college_going_rate']]
located_schools = gpd.GeoDataFrame(merged_df,geometry=gpd.points_from_xy(merged_df['Longitude'], merged_df['Latitude']), crs='EPSG:4326')

located_schools['college_going_rate'] = pd.to_numeric(located_schools['college_going_rate'], errors='coerce')
plt.figure(figsize=(8, 6))

ax = located_schools.plot(column='college_going_rate', cmap='magma', legend=False, markersize=5, figsize=(10, 6))


#  basemap
ctx.add_basemap(ax=ax, crs=located_schools.crs.to_string(), source=ctx.providers.CartoDB.Positron)

# colorbar
cbar = plt.colorbar(ax.collections[0], ax=ax, label='College Going Rate (%)')
ticks = np.linspace(0,100, num=5)
cbar.set_ticks(ticks)
plt.title("College going rate in California Public High Schools")
ax.tick_params(axis='both', which='both', length=0, labelsize=0)
plt.show()
# plt.savefig(f'{OUTPUT_DIR}/plots/college_going_rate_map.png', dpi=500)
# plt.close()
